# Visualize Excitation Rates
### Import libraries

In [1]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.notebook import tqdm

In [2]:
from utils import add_package_path

Paths

In [3]:
# Current Working Directory
cwd = os.getcwd()
# WORKSPACE
ws = os.environ['WORKSPACE_PATH']

In [4]:
prefix = '/home/zanardi/Workspace/ML/PyCOMET'
paths = {
    # PyCOMET source dir
    'pycomet': prefix + '/pycomet/',
    # SurQCT source dir
    'surqct': prefix + '/pycomet/examples/SurQCT/',
    # PyCOMET input module
    'inputs': prefix + '/run/SurQCT/O3_UMN/final/Inel/configs.ln/config.03/run.03/',
    # SurQCT input module
    'InputData': prefix + '/run/SurQCT/O3_UMN/final/Inel/configs.ln/datagen/'
}
add_package_path(paths)

In [5]:
# Output directory
out = cwd + '/Output/ExcitRates/Inel/final/c01.r01/'
if not os.path.exists(out):
    os.makedirs(out)

### Initialize PyCOMET model
Working case

In [6]:
os.chdir(paths['inputs'])

Import additional modules

In [7]:
from pycomet.utilities import utils
from pycomet.apps import functions as fun
# Inputs
import inputs
from inputs.general import flags

Compile model

In [8]:
# Load network architecture
net = fun._net(
    input_mod=inputs
)
# Finalize model
for k in ('train', 'save'):
    flags[k] = 0
flags['load'] = 2
model = fun._model(
    flags=flags,
    input_mod=inputs,
    net=net
)


[PyCOMET]: Initializing 'OpNN' network ...

[PyCOMET]: Building 'OpNN' network ...
Model: "SurQCT_OpNN"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 4)]          0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 4)]          0                                            
__________________________________________________________________________________________________
normalization (Normalization)   (None, 4)            9           input_1[0][0]                    
__________________________________________________________________________________________________
normalization_1 (Normalization) (None, 4)            9           input_2[0][0]                    
____

In [9]:
os.chdir(cwd)

### Initialize SurQCT parameters

Import additional modules

In [10]:
from surqct.src.chemical_systems import systems as syst
from surqct.src.rates_type.rates_type import RatesType
from surqct.src.utilities import utils
from surqct.src.utilities import const
from surqct import config as cfg
# SurQCT input module
import InputData

In [11]:
# Visualization libraries
# matplotlib = utils.import_matplotlib()
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']
%matplotlib qt

Inputs

In [12]:
SuffixesHDF5 = {
    'QCT':    '/Orig/',
    'SurQCT': ''
}
TTranVec    = [5000]
ValueMinMax = [1.e-14, None]
iIdxVec     = [0, 10, 110, 97, 2930, 3500, 4580, 6114]

Initializing QCT & SurQCT chemical systems

In [13]:
Systems = {}
for Identifier, SuffixHDF5 in SuffixesHDF5.items():
    # Initializing rates
    Systems[Identifier] = syst.get_system(
        NameLong=InputData.SystemName,
        KinSource=Identifier,
        ProcNames=tf.nest.flatten(InputData.RatesType),
        PathToHDF5Fld=cfg.paths.HDF5Dtb + '/' + Identifier + '/' + SuffixHDF5
    )
    if Identifier == 'QCT':
        # Reading rates
        Systems[Identifier].Processes.read_rates(TTranVec=TTranVec)
# Number of levels
NLevels = list(Systems.values())[0].Processes.CompInOut[InputData.RatesType]['In'][0].NLevels

Reading & predicting rates

In [14]:
Data = {}
# Loop over temperature
for TTran in TTranVec:
    TTranStr = str(int(TTran))
    Data[TTranStr] = {}
    # Loop over systems
    for Identifier, System in Systems.items():
        if Identifier == 'QCT':
            # Loop over indices
            Data[TTranStr][Identifier] = {
                str(iIdx): System.Processes.Rates[InputData.RatesType][TTranStr][iIdx] for iIdx in iIdxVec
            }
        elif Identifier == 'SurQCT':
            rates_type = RatesType(System, InputData)
            Data[TTranStr][Identifier] = {}
            # Loop over indices
            for iIdx in iIdxVec:
                xPred, yPred = rates_type.generate_predictiondata(
                    Model=model.net,
                    Columns=inputs.data.columns['x'],
                    TTranVec=[TTran],
                    iIdxVec=[iIdx]
                )
                Data[TTranStr][Identifier][str(iIdx)] = yPred.to_numpy().reshape(-1)


[SurQCT]: Generating Prediction data points ...
[SurQCT]:   Reading Variables ['log_EVib', 'log_ERot', 'log_VMax', 'log_rIn', 'log_rOut', 'log_rOut-rIn', 'rIn/rMin_min', 'log_rOut/rMin_min']
[SurQCT]:   Translational Temperature: T = 5000 K


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.42s/it]



[SurQCT]: Generating Prediction data points ...
[SurQCT]:   Reading Variables ['log_EVib', 'log_ERot', 'log_VMax', 'log_rIn', 'log_rOut', 'log_rOut-rIn', 'rIn/rMin_min', 'log_rOut/rMin_min']
[SurQCT]:   Translational Temperature: T = 5000 K


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.80it/s]



[SurQCT]: Generating Prediction data points ...
[SurQCT]:   Reading Variables ['log_EVib', 'log_ERot', 'log_VMax', 'log_rIn', 'log_rOut', 'log_rOut-rIn', 'rIn/rMin_min', 'log_rOut/rMin_min']
[SurQCT]:   Translational Temperature: T = 5000 K


...

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.22it/s]



[SurQCT]: Generating Prediction data points ...
[SurQCT]:   Reading Variables ['log_EVib', 'log_ERot', 'log_VMax', 'log_rIn', 'log_rOut', 'log_rOut-rIn', 'rIn/rMin_min', 'log_rOut/rMin_min']
[SurQCT]:   Translational Temperature: T = 5000 K


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.69it/s]



[SurQCT]: Generating Prediction data points ...
[SurQCT]:   Reading Variables ['log_EVib', 'log_ERot', 'log_VMax', 'log_rIn', 'log_rOut', 'log_rOut-rIn', 'rIn/rMin_min', 'log_rOut/rMin_min']
[SurQCT]:   Translational Temperature: T = 5000 K


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.08it/s]


### Plotting rates

In [15]:
def plot_prediction(
    Path,
    System,
    Data,
    iIdxVec,
    ValueMinMax,
    FigName=None
):
    fig, ax = plt.subplots(2, 2, sharex=True, sharey=True)

    Mol = System.Molecules[0]
    x, y = Mol.DiatData['EVib0'] - Mol.DissE, Mol.DiatData['ERot0']
    for i, iIdx in enumerate(iIdxVec):
        for j, (identifier, data) in enumerate(Data.items()):
            mappable = ax[i,j].scatter(
                x,
                y,
                c=data[str(iIdx)],
                cmap='jet',
                s=20,
                norm=matplotlib.colors.LogNorm(
                    vmin=ValueMinMax[0],
                    vmax=ValueMinMax[1]
                )
            )

    fig.add_subplot(111, frame_on=False)
    plt.tick_params(labelcolor='none', which='both', top=False, bottom=False, left=False, right=False)
    plt.xticks([])
    plt.yticks([])
    plt.xlabel('$\epsilon_{Vib}$ [eV]', labelpad=50)
    # plt.ylabel('$\epsilon_{Rot}$ [eV]', labelpad=60)
    fig.text(0., 0.41, '$\epsilon_{Rot}$ [eV]', va='center', ha='center', rotation='vertical', fontsize=plt.rcParams['axes.labelsize'])

    cbar = plt.colorbar(
        mappable,
        ax=ax.ravel().tolist(),
        orientation='horizontal',
        location='top',
        aspect=50
    )
    cbar.set_label(r'$k_{ij}^I$ [cm$^3$/s]', labelpad=30, size=30)
    cbar.ax.tick_params(labelsize=22)
    

    if FigName is None:
        FigName = 'diss3D'
    plt.savefig(Path + '/' + FigName + '.pdf')


In [16]:
ValueMinMax[1] = np.amax(tf.nest.flatten(Data))

In [17]:
# for i, TTran in enumerate(tqdm(TTranVec)):
#     TTranStr = str(int(TTran))
#     plot_prediction(
#         Path=out,
#         System=Systems['QCT'],
#         Data=Data[TTranStr],
#         iIdxVec=iIdxVec,
#         ValueMinMax=ValueMinMax,
#         FigName=f'T{TTranStr}K',
#     )

In [18]:
for i, TTran in enumerate(tqdm(TTranVec)):
    TTranStr = str(int(TTran))
    for iIdx in iIdxVec:
        TrueData = Data[TTranStr]['QCT'][str(iIdx)]
        PredData = Data[TTranStr]['SurQCT'][str(iIdx)]
        jIdxTrue = np.where(TrueData > 1.e-15)[0]
        jIdxPred = np.where(PredData > 1.e-15)[0]
        rates_type.plot_prediction3D(
            Path=out,
            iIdx=iIdx,
            TrueData=(jIdxTrue, TrueData[jIdxTrue]),
            PredData=(jIdxPred, PredData[jIdxPred])
        )

  0%|          | 0/1 [00:00<?, ?it/s]